In [0]:
def export_table_to_csv(table_name, volume_output_path):
    """
    Exporta uma tabela Delta para um arquivo CSV único e nomeado dentro de um Volume.
    Resolve o problema de colunas complexas (Arrays/Structs) que o CSV nativo não suporta.
    """
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, to_json
    spark = SparkSession.builder.getOrCreate()
    
    # --- PREPARAÇÃO DO NOME ---
    # Extrai apenas o nome da tabela, ignorando o catálogo e schema (ex: 'posts_creator')
    clean_table_name = table_name.split(".")[-1]
    final_filename = f"{clean_table_name}.csv"
    
    # 1. CARGA DOS DADOS
    df = spark.table(table_name)
    
    # 2. TRATAMENTO DE TIPOS COMPLEXOS (Conversão para JSON)
    # CSV não aceita listas (Arrays) ou objetos (Structs). 
    # Esta lógica varre o schema e converte automaticamente esses tipos para Strings no formato JSON.
    final_cols = [
        to_json(col(f.name)).alias(f.name) if str(f.dataType).startswith(("Struct", "Array")) 
        else col(f.name) for f in df.schema.fields
    ]
    
    # 3. GRAVAÇÃO TEMPORÁRIA
    # O Spark salva arquivos de forma distribuída. Usamos .coalesce(1) para forçar 
    # todos os dados a irem para um único "part-file", gerando um CSV só.
    temp_path = f"{volume_output_path}/_temp_{clean_table_name}"
    
    (df.select(final_cols)
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true") # Inclui o cabeçalho no CSV
        .option("sep", ";")       # Define ponto e vírgula como separador (padrão Excel BR)
        .csv(temp_path))
    
    # 4. RENOMEAÇÃO E ORGANIZAÇÃO (A "mágica" do arquivo único)
    # O Spark gera arquivos com nomes estranhos (ex: part-00000-abc-123.csv).
    # Aqui, listamos a pasta temporária para achar esse arquivo e movê-lo com o nome correto.
    files = dbutils.fs.ls(temp_path)
    part_file = [f.path for f in files if f.name.startswith("part-")][0]
    
    dest_path = f"{volume_output_path}/{final_filename}"
    dbutils.fs.cp(part_file, dest_path) # Copia o arquivo da pasta temporária para o destino final
    
    # 5. LIMPEZA (Sustentação: Não deixar lixo no storage)
    # Deleta a pasta temporária e os arquivos de metadados do Spark (.crc, _SUCCESS)
    dbutils.fs.rm(temp_path, True)
    
    print(f"✅ Exportação concluída: {final_filename}")
    return dest_path